In [42]:
# imports
import spacy
import json

In [43]:


# ### Entity and relation extraction functions
# def get_quantity(noun):
#     for child in noun.children:
#         if child.pos_ in ("DET", "NUM"):
#             return normalize_quantity(child, noun)
#     return "m"  # default

# ### Quantity normalization function
# def normalize_quantity(token, noun):
#     quantity_map = {
#         "ein": "1",
#         "eine": "1",
#         "jede": "1",
#         "jeder": "1",
#         "jedes": "1",
#         "mehrere": "n",
#         "viele": "n",
#         "alle": "n",
#         "einige": "n",
#         "kein": "0",
#         "keine": "0",
#     }

#     if token.pos_ == "NUM":
#         return token.text
#     elif noun.morph.get("Number") == ["Plur"]:
#         return "n"
#     elif token.pos_ == "DET":
#         text_lower = token.text.lower()
#         return quantity_map.get(text_lower, "m")
#     return "m"


# # Cardinality extraction function
# def get_cardinality(noun, modal=None):
#     """
#     Gibt (min, max) zurück basierend auf:
#     - Determinator
#     - Morphologie  
#     - Modalverb
#     - Numerale
#     """
#     det = None
#     num = None
    
#     for child in noun.children:
#         if child.pos_ == "DET":
#             det = child.text.lower()
#         elif child.pos_ == "NUM":
#             num = child.text

#     max_card = get_max(det, num, noun)
#     # min_card = get_min(det, num, noun, modal)
    
#     return (max_card)
#     return (dict(min=min_card), dict(max=max_card))

# # cardinality max function
# def get_max(det, num, noun):
#     if num:
#         return num  # z.B. "3"
#     if det in ("mehrere", "viele", "alle", "einige"):
#         return "n"
#     if noun.morph.get("Number") == ["Plur"]:
#         return "n"
#     return "1"  # Singular ohne Spezifikation

# # cardinality min function
# def get_min(det, num, noun, modal):
#     # Optionalität durch Modal
#     if modal in ("können", "dürfen"):
#         return "0"
#     if det in ("kein", "keine"):
#         return "0"
#     if num:
#         return num
#     #if det in ("mehrere", "einige"):
#     #    return "n"
#     return "1"  # Default


# def get_modal(sent):
#     """
#     Extrahiert Modalverb aus dem Satz falls vorhanden
#     """
#     for token in sent:
#         if token.pos_ == "AUX" and token.lemma_ in ("können", "müssen", "dürfen", "sollen", "wollen", "mögen"):
#             return token.lemma_
#     return None


In [44]:
# function declarations

# # Cardinality management functions
# def merge_cardinality(existing, new):
#     """n gewinnt immer gegen 1"""
#     if existing == "n" or new == "n":
#         return "n"
#     return new  # beide sind 1, neuer Wert überschreibt

# Entity management functions
def get_entity_id(noun, entities):
    global entity_id_counter
    key = noun.lemma_
    if key not in entities:
        entities[key] = {
            "id": f"e{entity_id_counter}",
            "type": noun.lemma_,
        }
        entity_id_counter += 1
    return entities[key]["id"]

# # Entity management functions
# def get_entity_id(noun, entities):
#     global entity_id_counter
#     key = noun.lemma_
#     new_card = get_cardinality(noun)
    
#     if key not in entities:
#         entities[key] = {
#             "id": f"e{entity_id_counter}",
#             "type": noun.lemma_,
#             "cardinality": new_card
#         }
#         entity_id_counter += 1
#     else:
#         # Konflikt auflösen: n bevorzugen
#         entities[key]["cardinality"] = merge_cardinality(
#             entities[key]["cardinality"], 
#             new_card
#         )
    
    return entities[key]["id"]

# Verb extraction function
def get_main_verb(sent):
    if [token for token in sent if token.pos_ == "VERB"]:
        return [token.lemma_ for token in sent if token.pos_ == "VERB"][0]
    root = sent.root

    # Falls ROOT ein Modal- oder Hilfsverb ist
    if root.pos_ in ["AUX", "VERB"]:
        for child in root.children:
            if child.dep_ in ["xcomp", "oc", "pd"] and child.pos_ == "VERB":
                return child.lemma_

    return root.lemma_

# Cardinality extraction function
def get_cardinality(token):
    for child in token.children:
        if child.lemma_ in ("ein", "eine"):
            return "1"
        if child.lemma_ in ("mehrere", "viele", "alle", "einige"):
            return "n"
        if child.dep_ == "det":
            # Morphologie auswerten
            number = child.morph.get("Number")
            if number == ["Plur"]:
                return "n"
            if number == ["Sing"]:
                return "1"
    # Fallback: Morphologie des Nomens selbst
    number = token.morph.get("Number")
    if number == ["Plur"]:
        return "n"
    return "1_fallback"


def merge_relations(relations):
    """
    Checks whether two relations have the same predicate.
    If subject/object are the same or swapped:
    - Merge cardinalities using max() (n > 1)
    - Delete duplicate
    """
    def card_max(a, b):
        """n wins over 1"""
        return "n" if "n" in (a, b) else "1"

    to_delete = set()

    for i, r1 in enumerate(relations):
        if i in to_delete:
            continue

        for j, r2 in enumerate(relations):
            if j <= i or j in to_delete:
                continue

            # checks if same predicate
            if r1["predicate"] != r2["predicate"]:
                continue

            same   = r1["subject"] == r2["subject"] and r1["object"] == r2["object"]
            swapped = r1["subject"] == r2["object"]  and r1["object"] == r2["subject"]

            if same:
                # if same, merge cardinalities directly
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["subject"]
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["object"]
                )
                to_delete.add(j)

            elif swapped:
                # if swapped, merge cardinalities crosswise
                r1["cardinality"]["subject"] = card_max(
                    r1["cardinality"]["subject"],
                    r2["cardinality"]["object"]     # crosswise merge
                )
                r1["cardinality"]["object"] = card_max(
                    r1["cardinality"]["object"],
                    r2["cardinality"]["subject"]    # crosswise merge
                )
                to_delete.add(j)

    return [r for i, r in enumerate(relations) if i not in to_delete]

# Entity to attribute
def collapse_weak_entities(entities, relations):
    """
    Removes weak entities (attribute candidates):
    - has exactly 1 relation
    - cardinality on its side is ‘1’
    - has no attributes of its own
        - is deleted, relation is deleted, its type is appended as an attribute to the other entity
    """
    to_delete_entities = set()
    to_delete_relations = set()

    # Index: entity_id → alle relations + auf welcher Seite sie steht
    from collections import defaultdict
    entity_relations = defaultdict(list)  # id → [(rel_index, side)]
    for i, rel in enumerate(relations):
        entity_relations[rel["subject"]].append((i, "subject"))
        entity_relations[rel["object"]].append((i, "object"))

    for key, entity in entities.items():
        eid = entity["id"]

        # Bedingung 1: genau eine Relation
        if len(entity_relations[eid]) != 1:
            continue

        rel_index, side = entity_relations[eid][0]
        rel = relations[rel_index]

        # Bedingung 2: Kardinalität auf ihrer Seite ist "1"
        if rel["cardinality"][side] != "1":
            continue

        # Bedingung 3: keine eigenen Attribute
        if entity.get("attributes"):
            continue

        # Andere Seite bestimmen
        other_side = "object" if side == "subject" else "subject"
        other_id = rel[other_side]

        # Anderen Entity-Key finden
        other_key = next((k for k, e in entities.items() if e["id"] == other_id), None)
        if other_key is None:
            continue

        # Typ der schwachen Entity als Attribut an die andere hängen
        entities[other_key].setdefault("attributes", []).append(entity["type"])

        to_delete_entities.add(key)
        to_delete_relations.add(rel_index)

    # Bereinigen
    for key in to_delete_entities:
        del entities[key]

    relations[:] = [r for i, r in enumerate(relations) if i not in to_delete_relations]

    return entities, relations
   

In [45]:
# variable declaration
nlp = spacy.load("de_core_news_lg")

# Limitations: Need same nouns and same verb in same relation. Change Synonyms and Homonymes.
# Perfect text example:
text = """
        Ein Kunde kann mehrere Bestellungen aufgeben.
        Eine Bestellung wird von genau einem Kunden aufgegeben.
        Eine Bestellung enthält mehrere Produkte.
        Ein Produkt ist in mehreren Bestellungen enthalten.
        Zu einem Kunden gehört genau eine Lieferadresse.
        Eine Lieferadresse gehört genau einem Kunden.
        Ein Produkt kann eine Beschreibung haben.
        """
# only nessessery information:
text = """
        Ein Kunde kann mehrere Bestellungen aufgeben.
        Eine Bestellung enthält mehrere Produkte, welche in mehreren Bestellungen enthalten sein können.
        Zu einem Kunden gehört genau eine Lieferadresse.
        Ein Produkt kann eine Beschreibung haben.
        """
#text = " ".join(text.split())
doc = nlp(text)

entities = {}
relations = []

entity_id_counter = 1


In [46]:
# Main - extraction loop

# Extract entities and relations
for sent in doc.sents:  # Loop through sentences
    
    # Extract subject, verb, and object
    subject = None
    subject_card = None
    obj = None
    object_card = None
    verb = get_main_verb(sent)
    attribute = None

    # Identify subject and object based on dependency labels
    for token in sent:
        if token.dep_ in ("sb", "nsubj"):
            subject = token
        elif token.dep_ in ("oa", "obj") or ((token.dep_ == "nk" or token.dep_ == "da") and token.pos_ == "NOUN"):
            obj = token

    # Only create a relation if we have a valid subject, verb, and object
    if subject and verb and obj:
        subj_id = get_entity_id(subject, entities)
        obj_id = get_entity_id(obj, entities)

        # Add relation with cardinality information
        # modal = get_modal(sent)

        if subject_card != {"n"}:
            subject_card = get_cardinality(subject)
        if object_card != {"n"}:
            object_card = get_cardinality(obj)

        relations.append({
            "subject": subj_id,
            "predicate": verb,
            "object": obj_id,
            "cardinality": {
                "subject": subject_card,
                "object": object_card
            }
})

# Merge duplicate relations
relations = merge_relations(relations)

# Collapse weak entities into attributes
entities, relations = collapse_weak_entities(entities, relations)

output = {
    "entities": list(entities.values()),
    "relations": relations
}

print(json.dumps(output, indent=2, ensure_ascii=False))
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)


{
  "entities": [
    {
      "id": "e1",
      "type": "Kunde",
      "attributes": [
        "Lieferadresse"
      ]
    },
    {
      "id": "e2",
      "type": "Bestellung"
    },
    {
      "id": "e3",
      "type": "welcher"
    },
    {
      "id": "e6",
      "type": "Beschreibung",
      "attributes": [
        "Produkt"
      ]
    }
  ],
  "relations": [
    {
      "subject": "e1",
      "predicate": "aufgeben",
      "object": "e2",
      "cardinality": {
        "subject": "1",
        "object": "n"
      }
    },
    {
      "subject": "e3",
      "predicate": "enthalten",
      "object": "e2",
      "cardinality": {
        "subject": "n",
        "object": "n"
      }
    }
  ]
}


In [47]:
for token in doc:
    print(token.text, token.dep_, token.pos_)


         dep SPACE
Ein nk DET
Kunde sb NOUN
kann ROOT AUX
mehrere nk DET
Bestellungen oa NOUN
aufgeben oc VERB
. punct PUNCT

         dep SPACE
Eine nk DET
Bestellung sb NOUN
enthält ROOT VERB
mehrere nk DET
Produkte oa NOUN
, punct PUNCT
welche sb PRON
in mo ADP
mehreren nk DET
Bestellungen nk NOUN
enthalten oc VERB
sein oc AUX
können rc AUX
. punct PUNCT

         dep SPACE
Zu op ADP
einem nk DET
Kunden nk NOUN
gehört ROOT VERB
genau mo ADV
eine nk DET
Lieferadresse sb NOUN
. punct PUNCT

         dep SPACE
Ein nk DET
Produkt sb NOUN
kann ROOT AUX
eine nk DET
Beschreibung oa NOUN
haben oc VERB
. punct PUNCT

         dep SPACE


In [48]:
import spacy
from spacy import displacy

text = nlp("Eine Bestellung enthält mehrere Produkte, welche in mehreren Bestellungen enthalten sein können.")

#displacy.serve(text, style="dep")

# tmp

In [49]:
tmp_doc = nlp("Eine Bestellung enthält mehrere Produkte.")
print("tmp_doc:", type(tmp_doc))
for token in tmp_doc:
    print(token.lemma_)
#    print(token.text, token.dep_, token.pos_)

tmp_doc: <class 'spacy.tokens.doc.Doc'>
ein
Bestellung
enthalten
mehrere
Produkt
--


In [50]:
def extract_kern(text):
    doc = nlp(text)
    result = {"subjekt": None, "verb": None, "objekt": None}

    hilfsverben = {"sein", "werden", "haben"}

    for token in doc:
        # Verb: ROOT, aber bei Hilfsverb → pd bevorzugen
        if token.dep_ == "ROOT":
            if token.lemma_ in hilfsverben:
                # Suche nach Prädikativ
                pd = next((c for c in token.children if c.dep_ == "pd"), None)
                result["verb"] = pd.lemma_ if pd else token.lemma_
            else:
                result["verb"] = token.lemma_

        # Subjekt
        if token.dep_ in ("sb", "nsubj") and token.pos_ in ("NOUN", "PROPN"):
            result["subjekt"] = token.lemma_

        # Direktes Objekt
        if token.dep_ in ("oa", "oc", "dobj") and token.pos_ in ("NOUN", "PROPN"):
            result["objekt"] = token.lemma_

        # Objekt in Präpositionalphrase (z.B. "in Bestellungen")
        if token.dep_ in ("mo", "pg") and token.pos_ == "ADP":
            for child in token.children:
                if child.pos_ in ("NOUN", "PROPN"):
                    result["objekt"] = child.lemma_

    return result

sätze = [
    "Ein Produkt ist in mehreren Bestellungen enthalten."
]

for satz in sätze:
    r = extract_kern(satz)
    print(f"{r['subjekt']} {r['verb']} {r['objekt']}")


Produkt enthalten Bestellung


In [51]:
sätze = [
    "Ein Kunde kann mehrere Bestellungen aufgeben.",
    "Eine Bestellung gehört genau einem Kunden.",
    "Eine Bestellung enthält mehrere Produkte.",
    "Ein Produkt ist in mehreren Bestellungen enthalten.",
]

for satz in sätze:
    doc = nlp(satz)
    for token in doc:
        if token.pos_ in ("NOUN", "PROPN"):
            card = get_cardinality(token)
            print(f"{token.text:<15} dep={token.dep_:<6} card={card}")
    print()

Kunde           dep=sb     card=1
Bestellungen    dep=oa     card=n

Bestellung      dep=sb     card=1
Kunden          dep=da     card=1

Bestellung      dep=sb     card=1
Produkte        dep=oa     card=n

Produkt         dep=sb     card=1
Bestellungen    dep=nk     card=n

